# 1. Generative AI Model Selection & Setup

### 1.1 Why Generative AI?

The supervised learning model built in Phase 1 (Random Forest) outputs a fit label — **Fit**, **Small**, or **Large**. While this is useful, a raw label is not particularly helpful to a real shopper. The goal of integrating Generative AI is to translate that prediction into a natural language explanation that is personalized, readable, and actionable.

For example, instead of showing a user `"Small"`, the system should be able to say:
> *"Based on your measurements, this item tends to run small for your body type. You may want to consider sizing up to a medium."*

This makes the system far more usable for non-technical end users.

### 1.2 Model Candidates Considered

We evaluated four potential models before selecting one:

| Model | Provider | Access | Strengths | Weaknesses |
|---|---|---|---|---|
| GPT-4o-mini | OpenAI | Paid API | Strong instruction following, reliable outputs, well-documented | Requires payment, not free |
| Gemini 1.5 Flash | Google | Free tier API | Generous free tier, good for short outputs | Free tier unavailable in Saudi Arabia |
| Claude Haiku | Anthropic | Paid API | Fast, high quality, globally accessible | Requires minimum credit purchase |
| LLaMA 3.1 8B (via Groq) | Meta / Groq | Free API | Fully free, no billing, fast inference, globally accessible | Less widely documented than OpenAI |

### 1.3 Selected Model: LLaMA 3.1 8B via Groq

We selected **LLaMA 3.1 8B served through Groq's API** for the following reasons:

- **Fully free**: No credit card or billing setup required. This makes it practical for us.
- **Regional accessibility**: Unlike Google's Gemini free tier, Groq's API works without quota restrictions in Saudi Arabia, which was a confirmed issue during our setup.
- **Sufficient output quality**: For short, structured text generation tasks like our prompt templates — which produce 2–4 sentence responses — LLaMA 3.1 8B performs well and produces clean, readable outputs.
- **Fast inference**: Groq's hardware (LPU-based) is notably faster than standard API providers, which is useful when running multiple test cases across four templates.

GPT-4o-mini remains the strongest model for this task in terms of output quality, but its paid requirement made it unsuitable as a primary choice for this project.

### 1.4 API Setup

The API key is stored in a `.env` file and loaded using `python-dotenv`. The key is **never hardcoded** in this notebook. See `api_config_template.py` for the full setup template.

Required libraries:
```
groq
python-dotenv
```

# 2. Prompt Template Design Documentation

We designed four prompt templates, each with a distinct strategy. The templates progressively incorporate more context, from a minimal baseline to a full measurement-and-cluster-informed guide. This design allows us to compare how much context the model actually needs to produce useful output.

All templates are saved as `.json` files in `/Generative_AI/prompts/`.

---
### Template 1 – Basic Prediction Explainer

**Template ID:** T1  
**Intended Use Case:** Minimal-context explanation of the fit label. This template assumes we have no user data available — only the raw prediction from the ML model.  

**Design Rationale:**  
This serves as our baseline. Before adding any user data, we want to understand how well the model can explain a fit prediction on its own. Many real scenarios involve incomplete user profiles, so a baseline that works with just a label is practically valuable. It also helps isolate the effect of adding context in later templates.

**Prompt Structure:**
```
A clothing size recommendation system predicted that a '{prediction}' fit label applies
to this purchase. In 2-3 sentences, explain what this means for the customer in simple,
friendly language. Do not suggest any action, just explain the prediction.
```

**Placeholders:**
- `{prediction}`: fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{ "prediction": "Small" }
```

**Example Output (expected):**
> *"The system predicted that this item runs small, meaning the size you selected may feel tighter or shorter than expected. This is common with certain brands whose sizing doesn't align with standard measurements. It's worth keeping this in mind when finalizing your choice."*

**Assumptions & Limitations:**  
No personalization — the output is the same for every user with the same label. It is informative but generic. Useful as a fallback when user data is unavailable.

---
### Template 2 – Measurement-Aware Personalized Advice

**Template ID:** T2  
**Intended Use Case:** Personalized advice that incorporates the customer's actual body measurements alongside the prediction.  

**Design Rationale:**  
The dataset contains several body-related measurements — height, bra size, cup size, and hip measurements — along with the clothing length and size ordered. These are the exact features our Random Forest model used to make the prediction, so passing them back to the language model gives it the context to explain *why* the prediction was made, not just what it is. This is the most natural upgrade from T1 and produces advice that feels genuinely tailored to the individual.

**Prompt Structure:**
```
A customer with the following measurements is shopping online:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}
- Cup size: {cup_size}
- Preferred clothing length: {length}
- Size ordered: {size}

Our ML model predicted the fit as '{prediction}' for the item they selected.
Based on these measurements and the prediction, give the customer a short, friendly
sizing recommendation (2-3 sentences). Be specific about their measurements.
```

**Placeholders:**
- `{height}` — customer height in cm (from `height_scaled`, reversed)
- `{hips}` — hip measurement in inches (from `hips_scaled`, reversed)
- `{bra_size}` — bra size e.g. `34`, `36` (from `bra size_scaled`, reversed)
- `{cup_size}` — cup size e.g. `B`, `C`, `D` (from `cup size_scaled`, reversed)
- `{length}` — preferred clothing length e.g. `just right`, `slightly long`, `very short` (from `length_scaled`, reversed)
- `{size}` — size ordered e.g. `7`, `24`, `25`, `33` (from `size_scaled`, reversed)
- `{prediction}` — fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "height": 165,
  "hips": 40,
  "bra_size": 36,
  "cup_size": "C",
  "length": "just right",
  "size": "24",
  "prediction": "Large"
}
```

**Example Output (expected):**
> *"Based on your measurements, the medium you ordered is predicted to run large — meaning it will likely feel loose around your hips and bust area. For a 36C with 40-inch hips at your height, sizing down to a small would probably give you a better fit. This is especially common in regular-length tops and dresses where the waist cut tends to be generous."*

**Assumptions & Limitations:**  
Requires that scaled features are inverse-transformed before being passed to the prompt, so the model receives interpretable values rather than normalized numbers. If any measurement fields are missing in a user's record, the prompt will degrade, a fallback to T1 should be used in that case.

---
### Template 3 – Cluster-Informed Contextual Advice

**Template ID:** T3  
**Intended Use Case:** Advice informed by the customer's cluster group identified during unsupervised learning, where clusters were derived from the same body measurement features used in the classifier.  

**Design Rationale:**  
The clustering step groups customers with similar combinations of height, hip measurements, bra/cup size, and clothing length preference. These groups capture fit patterns that individual measurements alone may not express clearly — for example, a cluster of customers who are tall with larger hip measurements and consistently report items running small in the hips. Passing the cluster's characteristic profile to the language model allows it to generate advice grounded in group-level patterns rather than just interpreting one individual's numbers in isolation.

**Prompt Structure:**
```
Based on body measurements and clothing preferences, this customer belongs to a
customer group characterized by: {cluster_description}.

Their selected item received a predicted fit of '{prediction}'.

Using this group profile, write a 2-3 sentence sizing suggestion that reflects
what customers with these characteristics commonly experience when shopping for
clothing online.
```

**Placeholders:**
- `{cluster_description}` - a human-readable summary of the cluster's defining features. Should reference the actual measurement features e.g. *"above-average height (170+ cm), larger hip measurements (42+ inches), and a preference for regular-length items"*
- `{prediction}` - fit label from ML model: `Fit`, `Small`, or `Large`

**Example Input:**
```json
{
  "cluster_description": "above-average height (170+ cm), larger hip measurements (42+ inches), bra size 36–38, and a preference for regular-length clothing",
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"Customers with your body profile — taller builds with fuller hips — often find that items run small through the hips and waist even when length fits well. Since this item was predicted to run small for you, sizing up by one would likely give a more comfortable fit in those areas. This is a pattern we commonly see for shoppers with a similar measurement profile."*

**Assumptions & Limitations:**  
The cluster description is written manually based on the centroid analysis done in Part A. Its quality depends directly on how well-separated and interpretable the clusters are. If clusters overlap significantly in feature space, the descriptions will be vague and the advice will not be meaningfully different from T2.

---
### Template 4 – Actionable Shopping Guide

**Template ID:** T4  
**Intended Use Case:** Full-context, actionable advice combining the customer's measurements, clothing category, quality rating, and the fit prediction into a practical shopping guide.  

**Design Rationale:**  
Templates 1–3 are explanatory. This template shifts toward telling the user what to *do*. It adds two fields that the others lack: `category` (the type of clothing item) and `quality` (the quality rating the customer associated with the item). Category matters because fit issues differ significantly across item types — a "Small" prediction in dresses has different implications than in tops or bottoms. Quality adds useful context because customers who rated quality lower may be experiencing a fit issue partly driven by poor construction rather than just sizing mismatch. The prompt explicitly requests three outputs: explanation, corrective size action, and a category-specific tip — to keep responses structured and comparable across test cases.

**Prompt Structure:**
```
A customer with the following profile ordered a clothing item:
- Height: {height} cm
- Hips: {hips} inches
- Bra size: {bra_size}, Cup size: {cup_size}
- Clothing length preference: {length}
- Size ordered: {size}
- Item category: {category}
- Quality rating given: {quality} out of 5

The ML model predicted fit as '{prediction}'.

Write a short, practical shopping guide (3-4 sentences) that:
1. Explains the fit prediction in the context of their measurements
2. Suggests what size to try instead (if the fit is not 'Fit')
3. Gives one practical tip specific to shopping for {category} items with their body profile

Keep the tone friendly and direct.
```

**Placeholders:**
- `{height}` — customer height in cm
- `{hips}` — hip measurement in inches
- `{bra_size}` — bra size number
- `{cup_size}` — cup size letter
- `{length}` — clothing length preference
- `{size}` — size ordered
- `{category}` — one of: `tops`, `bottoms`, `dresses`, `outerwear`, `wedding`, or `new` (derived from the one-hot encoded category features)
- `{quality}` — quality rating (from `quality_scaled`, reversed to original scale)
- `{prediction}` — fit label from ML model

**Example Input:**
```json
{
  "height": 170,
  "hips": 42,
  "bra_size": 36,
  "cup_size": "D",
  "length": "just right",
  "size": "38",
  "category": "dresses",
  "quality": 3,
  "prediction": "Small"
}
```

**Example Output (expected):**
> *"The large you ordered is predicted to run small for your hip and bust measurements, this dress likely pulls tightly across the hips and chest. We'd suggest trying an XL to give your proportions the room they need, particularly through the hips. When shopping for dresses with a fuller bust and hip, look for styles with an empire waist or wrap cut as they tend to be more accommodating than straight-cut fits."*

**Assumptions & Limitations:**  
The `category` value needs to be decoded from the one-hot encoded features before being inserted into the prompt (e.g., if `cat_dresses = 1`, pass `"dresses"`). The quality rating is used as soft context — the model may not always incorporate it meaningfully, which is worth noting during evaluation. As with T2, all scaled features must be inverse-transformed before use.

## 3. Implementation & API Integration Code (Safe Key Handling)

In this section, we connect our prompt templates to a Generative AI model using the Groq API.

To ensure safe key handling, the API key is not stored directly in the notebook or source code. Instead, it is stored in a `.env` file in the project root and loaded securely using the `python-dotenv` library. This prevents accidental exposure of sensitive information when sharing the repository.

We used the `llama-3.1-8b-instant` model to generate responses for our prompt templates.

In [39]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key securely from .env
load_dotenv()

api_key = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# Model used in this project
model_to_use = "llama-3.1-8b-instant"

In [40]:
try:
    response = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {"role": "user", "content": "Hello! I am testing the Groq API connection for my university project."}
        ]
    )
    print("API connection successful!")
    print(response.choices[0].message.content)

except Exception as e:
    print("Error:", e)

API connection successful!
The Groq API seems like an interesting project. Groq is an AI hardware company that provides a low-latency, general-purpose tensor processing unit (TPU). To assist you with testing the Groq API connection, I'll need more information about your project and the issues you're facing.

Can you please provide more details about your university project, such as:

1. What programming language are you using?
2. What specific API endpoints or features are you trying to connect to?
3. Have you gone through the Groq API documentation and set up authentication?
4. Are you encountering any errors or issues while trying to establish a connection?

Providing this information will help me better understand your requirements and assist you more effectively.


The successful response above confirms that the API connection is working correctly and can be reused in the following sections to test our prompt templates.

In [41]:
import json

def load_template(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

T1 = load_template("Generative_AI/prompts/template1.json")
T2 = load_template("Generative_AI/prompts/template2.json")
T3 = load_template("Generative_AI/prompts/template3.json")
T4 = load_template("Generative_AI/prompts/template4.json")

templates = [T1, T2, T3, T4]

print("Templates loaded successfully!")

Templates loaded successfully!


In [42]:
def build_prompt(template, data):
    return template["prompt"].format(**data)

In [43]:
def to_original_form(row):
    """
    Converts the input data into human-readable original form
    before sending it to the language model.
    """
    return {
        "height": row["height"],
        "hips": row["hips"],
        "bra_size": row["bra_size"],
        "cup_size": row["cup_size"],
        "length": row["length"],
        "size": row["size"],
        "category": row.get("category", "dress"),
        "quality": row.get("quality", 4),
        "prediction": row["prediction"],
        "cluster_description": row.get(
            "cluster_description",
            "customers with similar body proportions"
        )
    }

In [44]:
def generate_response(prompt):
    response = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [45]:
def run_template(template, row):
    data = to_original_form(row)
    prompt = build_prompt(template, data)
    output = generate_response(prompt)

    return {
        "template_id": template["template_id"],
        "template_name": template["template_name"],
        "prompt": prompt,
        "response": output
    }

In [46]:
sample = {
    "height": 165,
    "hips": 38,
    "bra_size": "34",
    "cup_size": "C",
    "length": "knee-length",
    "size": "M",
    "category": "dress",
    "quality": 4,
    "prediction": "Small",
    "cluster_description": "shorter customers who often find items slightly tight"
}

demo_result = run_template(T1, sample)
print("Pipeline executed successfully!")
print(demo_result["response"])

Pipeline executed successfully!
If the 'Small' fit label applies to your purchase, it means that the clothing is designed to fit a person with a smaller body size. This typically means a narrower width across the shoulders and chest, a shorter sleeve length, and a more fitted silhouette. The garment should skim your body closely without feeling too loose or too tight.


The pipeline above confirms that the system can securely connect to the API, load prompt templates, prepare user inputs in original form, and generate responses successfully. This implementation will be used in the next section for systematic testing and comparison across all four templates.

# 4. Testing framework & Output Comparison

In this section, we evaluate the performance of the four prompt templates using multiple test cases.

Each template is tested using different user profiles from the dataset. The generated outputs are analyzed based on qualitative and quantitative criteria including:

- Relevance to prediction
- Clarity and readability
- Level of detail
- Personalization
- Safety and correctness

This comparison helps us determine which prompt template performs best for our system.

In [47]:
test_cases = [
    {
        "height": 160,
        "hips": 36,
        "bra_size": "32",
        "cup_size": "B",
        "length": "short",
        "size": "S",
        "category": "dress",
        "quality": 3,
        "prediction": "Small",
        "cluster_description": "petite customers who often find items tight"
    },
    {
        "height": 170,
        "hips": 40,
        "bra_size": "36",
        "cup_size": "D",
        "length": "knee-length",
        "size": "M",
        "category": "dress",
        "quality": 4,
        "prediction": "Fit",
        "cluster_description": "average body type customers with balanced proportions"
    },
    {
        "height": 175,
        "hips": 44,
        "bra_size": "38",
        "cup_size": "DD",
        "length": "long",
        "size": "L",
        "category": "dress",
        "quality": 5,
        "prediction": "Large",
        "cluster_description": "taller customers who often find items loose"
    }
]

In [48]:
results = []

for case_id, case in enumerate(test_cases):
    for t in templates:
        res = run_template(t, case)

        results.append({
            "case_id": case_id,
            "template": t["template_name"],
            "response": res["response"]
        })

In [49]:
for r in results:
    print(f"\nCase {r['case_id']} - {r['template']}")
    print(r["response"])


Case 0 - Basic Prediction Explainer
The 'Small' label suggests that this size will fit you snugly, but still comfortably, with some room to move. You can expect a fitted style with a slightly closer fit around the body. This size is ideal for individuals who prefer a more streamlined look but still want to feel relaxed.

Case 0 - Measurement-Aware Personalized Advice
Based on your height of 160 cm and preferred short clothing length, our model suggests the fit is 'Small' for the item you've selected, which aligns with the size you originally ordered (S). However, considering your hip measurement is on the larger side at 36 inches, we recommend checking the pant inseam to ensure it fits comfortably with the short length.

Case 0 - Cluster-Informed Contextual Advice
Based on your petite frame and tendency to find items tight, we recommend sizing up to our 'Medium' for a more comfortable fit. This approach often helps you achieve the desired fit without feeling restrictive or constricted

# 5. Analysis: Qualitative & Quantative Results
## Qualitative Analysis

### Template 1 – Basic Prediction Explainer
The responses were clear but generic. The template explains the prediction but lacks personalization and does not provide actionable advice.We also observed that this Template  sometimes misinterpreted the prediction label (e.g., treating "Small" as a body description instead of a fit label), which reduces its reliability.

### Template 2 – Measurement-Aware Personalized Advice
This template produced more personalized responses by incorporating user measurements. The advice was clearer and more relevant to the user's profile.

### Template 3 – Cluster-Informed Contextual Advice
This template leveraged cluster information effectively, providing context-aware suggestions. However, its performance depended on the quality of the cluster description.

### Template 4 – Actionable Shopping Guide
This template generated the most practical and helpful responses. It provided explanations, size recommendations, and useful shopping tips, making it the most complete.
## Quantitative Analysis

We compared the templates based on response length and level of detail.

- Template 1 produced the shortest responses
- Template 4 produced the longest and most detailed responses
- Templates 2 and 3 showed moderate response length with improved personalization

Overall, longer responses tended to provide more useful and actionable insights.

# 6. Best Prompt Selection & Justification

# 7. Integration Plan for Final System

# 8. Ethical Considerations & Limitations